# Prediksi Harga Mobil Bekas — Training Notebook

**Kasus**: Regresi — memprediksi harga mobil bekas berdasarkan spesifikasi.

- **Fitur (X)**: Brand, Year, Engine Size, Fuel Type, Transmission, Mileage, Condition, Model
- **Target (y)**: Price
- **Algoritma**: Random Forest Regressor vs Linear Regression
- **Dataset**: Kaggle - Car Price Prediction (zafarali27)
  https://www.kaggle.com/datasets/zafarali27/car-price-prediction

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

## 2. Load Dataset

In [2]:
df = pd.read_csv("data/car_price_prediction.csv")
print("Shape:", df.shape)
df.head()

Shape: (2500, 10)


,Car ID,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,1,Tesla,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X
1,2,BMW,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series
2,3,Audi,2013,4.5,Electric,Manual,181601,New,44402.61,A4
3,4,Tesla,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y
4,5,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang


## 3. Eksplorasi & Pembersihan Data (Data Cleaning)

Mengecek missing value, duplikat, dan outlier pada data numerik.

In [3]:
print("Missing value per kolom:")
print(df.isnull().sum())
print("\nJumlah baris duplikat:", df.duplicated().sum())

Missing value per kolom:
Car ID          0
Brand           0
Year            0
Engine Size     0
Fuel Type       0
Transmission    0
Mileage         0
Condition       0
Price           0
Model           0
dtype: int64

Jumlah baris duplikat: 0


In [4]:
# Cek outlier dengan metode IQR pada kolom numerik
numeric_cols_check = ["Year", "Engine Size", "Mileage", "Price"]
for col in numeric_cols_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outlier = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {n_outlier} outlier terdeteksi (batas [{lower:.2f}, {upper:.2f}])")

Year: 0 outlier terdeteksi (batas [1985.50, 2037.50])
Engine Size: 0 outlier terdeteksi (batas [-1.55, 8.45])
Mileage: 0 outlier terdeteksi (batas [-159407.00, 457229.00])
Price: 0 outlier terdeteksi (batas [-41486.59, 146233.60])


In [5]:
# Fungsi untuk menghapus outlier berbasis IQR (dijalankan berjaga-jaga,
# meskipun pada dataset ini tidak ditemukan outlier signifikan)
def remove_outliers_iqr(data, cols):
    data = data.copy()
    for col in cols:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        data = data[(data[col] >= lower) & (data[col] <= upper)]
    return data

df_clean = remove_outliers_iqr(df, ["Year", "Engine Size", "Mileage", "Price"])
df_clean = df_clean.drop_duplicates()
print("Shape sebelum:", df.shape, "| Shape sesudah cleaning:", df_clean.shape)

Shape sebelum: (2500, 10) | Shape sesudah cleaning: (2500, 10)


In [6]:
# Kolom "Car ID" hanya identifier, tidak relevan sebagai fitur prediktif
df_clean = df_clean.drop(columns=["Car ID"])
df_clean.head()

,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,Tesla,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X
1,BMW,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series
2,Audi,2013,4.5,Electric,Manual,181601,New,44402.61,A4
3,Tesla,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y
4,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang


## 4. Feature Engineering

- Fitur numerik: `Year`, `Engine Size`, `Mileage` -> **Feature Scaling** (StandardScaler)
- Fitur kategorikal: `Brand`, `Fuel Type`, `Transmission`, `Condition`, `Model` -> **Feature Encoding** (One-Hot Encoding)

Kedua proses ini digabung dalam satu `ColumnTransformer` supaya urutan transformasi
saat training dan saat inferensi selalu konsisten (sesuai poin penting di rubrik tugas).

In [7]:
target_col = "Price"
numeric_features = ["Year", "Engine Size", "Mileage"]
categorical_features = ["Brand", "Fuel Type", "Transmission", "Condition", "Model"]

X = df_clean[numeric_features + categorical_features]
y = df_clean[target_col]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

## 5. Split Data (Train/Test)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (2000, 8) Test: (500, 8)


## 6. Melatih Minimal Dua Algoritma

Preprocessing (scaling + encoding) dijadikan satu `Pipeline` bersama model,
sehingga objek scaler & encoder otomatis ikut tersimpan saat model di-*joblib.dump*.

In [9]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

trained_pipelines = {}
results = []

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe

    y_pred = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

## 7. Evaluasi & Perbandingan Model

In [10]:
results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

            Model          MAE         RMSE        R2
Linear Regression 23877.144632 27794.415064 -0.019769
    Random Forest 24468.079399 28367.437479 -0.062251


**Catatan hasil**: Model dengan R2 tertinggi dan RMSE/MAE terendah dipilih sebagai model
terbaik. Pada dataset sintetis ini, hubungan antar fitur dan harga cenderung acak/tanpa
pola kuat, jadi nilai R2 kedua model bisa rendah/mendekati — itu wajar dan tidak
menggugurkan validitas alur (fokus tugas adalah kelengkapan alur, bukan akurasi tertinggi).

In [11]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_pipelines[best_model_name]
print("Model terbaik:", best_model_name)

Model terbaik: Linear Regression


## 8. Simpan Model & Objek Pendukung

Karena scaler & encoder sudah menyatu di dalam `Pipeline`, satu file `model.pkl` ini
sudah mencakup seluruh objek pendukung (tidak perlu file scaler/encoder terpisah).
Kita juga simpan daftar opsi kategori (untuk dropdown di aplikasi) dan metrik evaluasi.

In [12]:
import os
os.makedirs("model", exist_ok=True)

joblib.dump(best_pipeline, "model/model.pkl")

feature_options = {col: sorted(df_clean[col].unique().tolist()) for col in categorical_features}
feature_options["numeric_ranges"] = {
    col: {"min": float(df_clean[col].min()), "max": float(df_clean[col].max())}
    for col in numeric_features
}
feature_options["best_model_name"] = best_model_name
feature_options["numeric_features"] = numeric_features
feature_options["categorical_features"] = categorical_features

with open("model/feature_options.json", "w") as f:
    json.dump(feature_options, f, indent=2)

results_df.to_csv("model/model_comparison.csv", index=False)

print("Tersimpan: model/model.pkl, model/feature_options.json, model/model_comparison.csv")

Tersimpan: model/model.pkl, model/feature_options.json, model/model_comparison.csv


## 9. Uji Cepat Inferensi (sanity check)

In [13]:
sample = pd.DataFrame([{
    "Year": 2018,
    "Engine Size": 2.5,
    "Mileage": 45000,
    "Brand": "Toyota",
    "Fuel Type": "Petrol",
    "Transmission": "Automatic",
    "Condition": "Used",
    "Model": "Civic",
}])

loaded_pipeline = joblib.load("model/model.pkl")
pred = loaded_pipeline.predict(sample)
print("Prediksi harga:", round(float(pred[0]), 2))

Prediksi harga: 48893.1
